In [27]:
from langchain_google_genai import GoogleGenerativeAI
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS   
from langgraph.graph import END, START, StateGraph, add_messages
from langchain_core.messages import BaseMessage
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

In [3]:
load_dotenv()
llm = GoogleGenerativeAI(model="gemini-3.5-flash")
response = llm.invoke("Hello")
print(response)

Hello! How can I help you today?


In [26]:
document_loader = DirectoryLoader("./books", glob="*.pdf", loader_cls=PyPDFLoader)
docs = document_loader.load()

sample_docs = docs[:3]
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100, separators=["\n\n", "\n", " ", ""])
sample_chunks = text_splitter.split_documents(sample_docs)
print(f"Total sample chunks: {len(sample_chunks)}")

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store = FAISS.from_documents(
    documents=sample_chunks, embedding=embeddings
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("FAISS vector store and retriever are ready")


Total sample chunks: 11
FAISS vector store and retriever are ready


In [28]:
class CRAGState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    question: str
    documents: list
    filtered_docs: list
    verdict: Literal["correct", "incorrect", "ambiguous"]
    web_search: bool
    generation: str